# Task 04A-E — guarded A100 study
Run in order without restarting. The full study is permitted only when the committed CPU smoke gate says `AUTHORIZE_FULL`. This notebook never pushes to GitHub.

In [ ]:
REPO_URL = 'https://github.com/PaulsonLab/energy-inference-bo.git'
REPO_REF = 'main'  # or a full commit SHA reachable from main
RUN_FULL = False
OUTPUT_DIR = '/content/energy-inference-bo/artifacts/task04ae/full'

In [ ]:
import os, pathlib, shutil, subprocess, sys
repo = pathlib.Path('/content/energy-inference-bo')
if repo.exists():
    shutil.rmtree(repo)
subprocess.run(['git','clone','--branch','main',REPO_URL,str(repo)],check=True)
subprocess.run(['git','fetch','origin','main'],cwd=repo,check=True)
if REPO_REF != 'main':
    subprocess.run(['git','merge-base','--is-ancestor',REPO_REF,'origin/main'],cwd=repo,check=True)
    subprocess.run(['git','checkout','--detach',REPO_REF],cwd=repo,check=True)
sha=subprocess.check_output(['git','rev-parse','HEAD'],cwd=repo,text=True).strip()
print('Git SHA:',sha)
subprocess.run([sys.executable,'-m','pip','install','-q','uv'],check=True)
subprocess.run([sys.executable,'-m','uv','sync','--locked','--group','dev'],cwd=repo,check=True)
scientific_python=str(repo/'.venv/bin/python')
scientific_env=os.environ.copy(); scientific_env['MPLBACKEND']='Agg'; scientific_env['MPLCONFIGDIR']='/tmp/matplotlib-task04ae'; scientific_env['PYTHONPATH']=str(repo/'src')

In [ ]:
probe="import json,torch; print(json.dumps({'torch':torch.__version__,'cuda':torch.cuda.is_available(),'device':torch.cuda.get_device_name(0) if torch.cuda.is_available() else None}))"
subprocess.run([scientific_python,'-c',probe],cwd=repo,env=scientific_env,check=True)
gpu=subprocess.check_output([scientific_python,'-c',"import torch; print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')"],cwd=repo,env=scientific_env,text=True).strip()
assert 'A100' in gpu, f'An A100 runtime is required for the approved full profile; got {gpu!r}'
subprocess.run([scientific_python,'-m','pytest','-q'],cwd=repo,env=scientific_env,check=True)
gate_path=repo/'results/task04ae/smoke/gate_status.json'
assert gate_path.exists(), 'No reviewed Task 04A-E smoke gate is committed.'
import json
smoke_gate=json.loads(gate_path.read_text())
print(json.dumps(smoke_gate,indent=2))

In [ ]:
if not RUN_FULL:
    print('Full run disabled. Set RUN_FULL=True only after the reviewed smoke authorizes it.')
else:
    assert smoke_gate.get('authorize_full') and smoke_gate.get('decision') == 'AUTHORIZE_FULL', 'Committed smoke gate does not authorize the full study.'
    subprocess.run([scientific_python,'-m','energy_bo.experiments.run_task04ae','--profile','full','--device','cuda','--output-dir',OUTPUT_DIR],cwd=repo,env=scientific_env,check=True)

In [ ]:
if RUN_FULL:
    import json, platform, zipfile
    output=pathlib.Path(OUTPUT_DIR)
    manifest={'git_sha':sha,'python':platform.python_version(),'gpu':gpu,'command':'--profile full --device cuda','smoke_gate':smoke_gate}
    (output/'colab_manifest.json').write_text(json.dumps(manifest,indent=2)+'\n')
    archive=pathlib.Path('/content/task04ae_full_outputs.zip')
    with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as handle:
        for path in output.rglob('*'):
            if path.is_file(): handle.write(path,path.relative_to(output.parent))
    from google.colab import files
    files.download(str(archive))

After download, extract the ZIP locally under `artifacts/task04ae/full/`. Keep raw files ignored and request an audit before copying compact evidence into `results/task04ae/full/`.